# Statistics (Design 9 and 13; addenda J, O5, P5, R5; decisions D290 to D299)

This notebook only calls `scripts/analysis.py` and `scripts/stats_checks.py`. Runtime: **CPU**. No API, no GPU.

- **Cell 0**: Drive, the two clones, packages, one PASS or FAIL line per input (`stats_checks.py pins`), the D172 count.
- **Cell 1**: the gate-E self test and `check-inputs --mode real`, from the detached gate-E checkout.
- **Cell 2**: the primary run: Cell 8 of `notebooks/analysis.ipynb` with the real paths, unchanged gate-E code.
  The lexicon sensitivity pass (S9, `human/HT-5/lexicon_sensitivity.csv`) is a pre-specified stage of this run.
- **Cell 3**: the primary run against the values known before it; the key estimates.
- **Cell 4**: the D279 sensitivity pass (D290), from main, with `--deviation`; its own folder `sens_D279_<UTC>/`.
- **Cell 5**: the D279 pass against the values known before it; S11.
- **Cell 6**: copy the aggregate tables and figures to repo `results/stats/` and commit. Switched off until Claude
  has read the outputs.

Printouts hold ids, counts, hashes and estimates only. Run the cells in order; each cell stops on a FAIL.

In [ ]:
# Cell 0: settings, Drive, the two clones, packages, and one PASS or FAIL line per input.
DRIVE = "/content/drive/MyDrive/BanglaIndoorVQA"
MAIN = "/content/banglaindoorvqa-main"     # main: the D290 analysis.py and stats_checks.py
GATE_E = "/content/bivqa_gateE"            # detached checkout of the tag gate-E: the primary run
GH_REPO = "blkeyd/banglaindoorvqa-main"
OUT = DRIVE + "/results/stats"             # real output only (D230); never placeholder/
HT3_DIR = DRIVE + "/human/HT-3/score_final_20261023T061119Z"
SCORED = DRIVE + "/scored/main/*_scored.jsonl"
LOGS = OUT + "/logs"

import datetime, glob, os, subprocess, sys
from google.colab import drive, userdata
drive.mount("/content/drive")
TOKEN = userdata.get("GITHUB_TOKEN")
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"   # no __pycache__ inside a Git tree


def _mask(s):
    return s.replace(TOKEN, "***") if TOKEN else s


def git(*args):
    r = subprocess.run(["git"] + list(args), capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(_mask("git %s failed:\n%s%s" % (" ".join(args), r.stdout, r.stderr)))
    return r.stdout.strip()


def run(cmd, cwd, name):
    """Run a script, show every line it prints, and keep the same lines in Drive results/stats/logs/."""
    os.makedirs(LOGS, exist_ok=True)
    log = "%s/%s_%s.log" % (LOGS, name, datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
    env = dict(os.environ, PYTHONDONTWRITEBYTECODE="1", PYTHONUNBUFFERED="1", MPLBACKEND="Agg")
    print("$ cd %s && %s" % (cwd, " ".join(cmd)))
    with open(log, "w", encoding="utf-8") as fh:
        p = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1)
        for line in p.stdout:
            print(line, end="")
            fh.write(line)
        rc = p.wait()
    print("exit code %d; log %s" % (rc, log))
    return rc


url = "https://x-access-token:%s@github.com/%s.git" % (TOKEN, GH_REPO)
if not os.path.isdir(MAIN + "/.git"):
    git("clone", "-q", url, MAIN)
else:
    git("-C", MAIN, "remote", "set-url", "origin", url)
    git("-C", MAIN, "checkout", "-q", "main")
    git("-C", MAIN, "fetch", "-q", "--tags", "origin")
    git("-C", MAIN, "merge", "-q", "--ff-only", "origin/main")
git("-C", MAIN, "fetch", "-q", "--tags", "origin")
git("-C", MAIN, "remote", "set-url", "origin", "https://github.com/%s.git" % GH_REPO)   # no token left in .git
if not os.path.isdir(GATE_E + "/.git"):
    git("clone", "-q", MAIN, GATE_E)       # a local clone of main: the tags come along
git("-C", GATE_E, "fetch", "-q", "--tags", "origin")
git("-C", GATE_E, "checkout", "-q", "--detach", "gate-E")
print(git("-C", MAIN, "log", "-1", "--format=main HEAD %h %cI"))
print(git("-C", GATE_E, "log", "-1", "--format=gate-E checkout HEAD %h %cI"))
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sacrebleu", "statsmodels", "transformers==5.18.0"],
                   capture_output=True, text=True)
print("pip install:", "done" if r.returncode == 0 else (r.stdout[-1500:] + r.stderr[-1500:]))
rc0 = run([sys.executable, MAIN + "/scripts/stats_checks.py", "pins", "--drive", DRIVE, "--main", MAIN,
           "--gate-e", GATE_E, "--ht3-dir", HT3_DIR], "/content", "cell0_pins")
assert rc0 == 0, "Cell 0: a FAIL above. Stop and send the FAIL lines to Claude."

In [ ]:
# Cell 1: the gate-E self test, then check-inputs in real mode, both from the detached gate-E checkout.
REAL = ["--drive", DRIVE, "--scored", SCORED, "--ht3-dir", HT3_DIR,
        "--ht4", DRIVE + "/human/HT-4/ht4_labels.csv",
        "--ht5", DRIVE + "/human/HT-5/ht5_audit.csv",
        "--sensitivity", DRIVE + "/human/HT-5/lexicon_sensitivity.csv",
        "--tokens", DRIVE + "/results/stats_inputs/token_counts.csv"]
rc1a = run([sys.executable, "scripts/analysis.py", "selftest"], GATE_E, "cell1_selftest")
rc1b = run([sys.executable, "scripts/analysis.py", "check-inputs", "--mode", "real"] + REAL, GATE_E,
           "cell1_check_inputs")
print("gate-E checkout clean:", git("-C", GATE_E, "status", "--porcelain") == "")
assert rc1a == 0 and rc1b == 0, "Cell 1: a FAIL above. Stop and send the FAIL lines to Claude."

In [ ]:
# Cell 2: the primary run, unchanged gate-E code (Cell 8 of notebooks/analysis.ipynb with the real paths).
# 10,000 resamples, seed 42. About 10 to 20 minutes on CPU (BERTScore downloads its model once).
# Output: Drive results/stats/run_<UTC>/ (figures, tables, supplement S1 to S10, report, numbers, manifest, checks).
before = set(glob.glob(OUT + "/run_*"))
rc2 = run([sys.executable, "scripts/analysis.py", "run", "--mode", "real"] + REAL + ["--out", OUT], GATE_E, "cell2_run")
new_runs = sorted(set(glob.glob(OUT + "/run_*")) - before)
print("new run folder:", new_runs[-1] if new_runs else "none")
print("gate-E checkout clean after the run:", git("-C", GATE_E, "status", "--porcelain") == "")
assert rc2 == 0 and new_runs, "Cell 2: the run did not finish cleanly. Send the last 20 lines above to Claude."

In [ ]:
# Cell 3: the primary run against the values known before it (addenda O2, P2, P5, R; D303), then the key estimates.
rc3 = run([sys.executable, MAIN + "/scripts/stats_checks.py", "verify", "--drive", DRIVE, "--main", MAIN,
           "--run", "latest"], "/content", "cell3_verify")
print("\nSend Claude: this printout, then checks.txt and analysis_report.md of the run folder.")

In [ ]:
# Cell 4: the D279 sensitivity pass (D290) from main, beside the primary run, with the same resamples.
# About 3 to 5 minutes. Output: Drive results/stats/sens_D279_<UTC>/ (S11, d279_report.md, numbers, manifest, checks).
DEVIATION = ("D290: D279 sensitivity pass (sens-d279) added to scripts/analysis.py after the tag gate-E; "
             "the primary run is the unchanged gate-E code")
primary = sorted(glob.glob(OUT + "/run_*"))[-1]
print("primary run:", primary)
rc4 = run([sys.executable, "scripts/analysis.py", "sens-d279", "--mode", "real", "--drive", DRIVE, "--scored", SCORED,
           "--primary-run", primary, "--deviation", DEVIATION, "--out", OUT], MAIN, "cell4_d279")
print("main clone clean after the pass:", git("-C", MAIN, "status", "--porcelain") == "")
assert rc4 == 0, "Cell 4: a FAIL above. Send the FAIL lines to Claude."

In [ ]:
# Cell 5: the D279 pass against the values known before it (addendum O2), then S11.
rc5 = run([sys.executable, MAIN + "/scripts/stats_checks.py", "verify", "--drive", DRIVE, "--main", MAIN,
           "--d279", "latest", "--no-estimates"], "/content", "cell5_verify")
print("\nSend Claude: this printout and d279_report.md of the sens_D279 folder.")

In [ ]:
# Cell 6: copy the aggregate tables and figures of both folders into repo results/stats/ and commit (addendum J4).
# Only folders whose newest verify has 0 FAIL are copied; numbers JSON, per-item files and TEXT_ files stay in Drive.
# Switched off until Claude has read the outputs (the close of the Statistics chat).
STAGE_AND_COMMIT = False
GIT_NAME, GIT_EMAIL = "", ""               # your GitHub name and email for the commit
if STAGE_AND_COMMIT:
    assert GIT_NAME and GIT_EMAIL, "Fill GIT_NAME and GIT_EMAIL first"
    rc6 = run([sys.executable, MAIN + "/scripts/stats_checks.py", "stage", "--drive", DRIVE, "--main", MAIN,
               "--run", "latest", "--d279", "latest"], "/content", "cell6_stage")
    assert rc6 == 0, "Cell 6: stage refused a folder; see above."
    git("-C", MAIN, "add", "results/stats")
    print(git("-C", MAIN, "status", "--short"))
    git("-C", MAIN, "-c", "user.name=" + GIT_NAME, "-c", "user.email=" + GIT_EMAIL, "commit", "-q", "-m",
        "Statistics: aggregate tables and figures of the primary run and the D279 pass (addendum Q)")
    git("-C", MAIN, "push", "-q", url, "HEAD:main")
    print(git("-C", MAIN, "log", "-1", "--format=committed and pushed %h %cI"))